In [1]:
import os
import logging
from typing import Dict, List, Any, TypedDict
from dataclasses import dataclass
import json

from langchain_groq import ChatGroq
from langchain.schema import HumanMessage
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from sentence_transformers import SentenceTransformer
import numpy as np

from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState
from typing import Optional


load_dotenv()

e:\Crail 2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-70b-versatile")
QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", "6334"))
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "Crail_data")


In [4]:
@dataclass
class CurrentUser:
    """User information structure"""
    user_id: int
    name: str
    email: str

In [5]:


class EmailResponseState(TypedDict):
    """State structure for the LangGraph flow"""
    user_email: str
    current_user: CurrentUser
    email_summary: str
    search_results: List[Dict[str, Any]]
    response_email: Optional[Dict[str, str]]
    validation_score: int
    error: str


In [6]:
class EmailResponseFlow:
    """Main class for handling email response generation flow"""
    
    def __init__(self):
        """Initialize the email response flow components"""
        self.llm = self._init_llm()
        self.qdrant_client = self._init_qdrant()
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.graph = self._build_graph()
    
    def _init_llm(self) -> ChatGroq:
        """Initialize ChatGroq LLM"""
        return ChatGroq(
            model=GROQ_MODEL,
            api_key=GROQ_API_KEY,
            temperature=0.5,
            max_tokens=5000,
            top_p=0.95,
            frequency_penalty=0,
            presence_penalty=0,
            stop=None,
        )
    
    def _init_qdrant(self) -> QdrantClient:
        """Initialize Qdrant client"""
        try:
            client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
            logger.info(f"Connected to Qdrant at {QDRANT_HOST}:{QDRANT_PORT}")
            return client
        except Exception as e:
            logger.error(f"Failed to connect to Qdrant: {e}")
            raise
    
    def _build_graph(self) -> StateGraph:
        """Build the LangGraph workflow"""
        graph = StateGraph(EmailResponseState)
        
        # Add nodes
        graph.add_node("summarize_email", self.summarize_email_intent)
        graph.add_node("search_qdrant", self.search_knowledge_base)
        graph.add_node("generate_response", self.generate_email_response)
        graph.add_node("validate_response", self.validate_email_response)
        
        # Define edges
        graph.set_entry_point("summarize_email")
        graph.add_edge("summarize_email", "search_qdrant")
        graph.add_edge("search_qdrant", "generate_response")
        graph.add_edge("generate_response", "validate_response")
        graph.add_edge("validate_response", END)
        
        return graph.compile()
    
    def summarize_email_intent(self, state: EmailResponseState) -> EmailResponseState:
        """Step 1: Summarize the main intent of the incoming email"""
        logger.info("Step 1: Summarizing email intent")
        
        try:
            prompt = f"""
            Analyze the following email and extract the main question, request, or intent.
            Provide a clear, concise summary that captures what the sender is asking for or needs help with.
            
            Email content:
            {state['user_email']}
            
            Please provide only the summary of the main intent, without any additional commentary.
            """
            
            response = self.llm.invoke([HumanMessage(content=prompt)])
            email_summary = response.content.strip()
            
            logger.info(f"Email summary: {email_summary}")
            
            return {
                **state,
                "email_summary": email_summary
            }
            
        except Exception as e:
            logger.error(f"Error in summarize_email_intent: {e}")
            return {
                **state,
                "error": f"Failed to summarize email: {str(e)}"
            }
    
    def search_knowledge_base(self, state: EmailResponseState) -> EmailResponseState:
        """Step 2: Search Qdrant for relevant documents"""
        logger.info("Step 2: Searching knowledge base")
        
        try:
            # Create embedding for the email summary
            query_embedding = self.encoder.encode(state['email_summary']).tolist()

            user_filter = Filter(
                must=[
                    FieldCondition(
                        key="user_id",
                        match=MatchValue(value=state['current_user'].user_id,)
                    )
                ]
            )
            
            # Search Qdrant
            search_results = self.qdrant_client.search(
                collection_name=COLLECTION_NAME,
                query_vector=query_embedding,
                query_filter=user_filter, 
                limit=5,
                with_payload=True,
                with_vectors=False
            )
            
            # Format results
            formatted_results = []
            for result in search_results:
                formatted_results.append({
                    "score": result.score,
                    "data": result.payload.get("data", ""),
                    "metadata": result.payload.get("metadata", {})
                })
            
            logger.info(f"Found {len(formatted_results)} relevant documents")
            
            return {
                **state,
                "search_results": formatted_results
            }
            
        except Exception as e:
            logger.error(f"Error in search_knowledge_base: {e}")
            return {
                **state,
                "search_results": [],
                "error": f"Failed to search knowledge base: {str(e)}"
            }
    
    def generate_email_response(self, state: EmailResponseState) -> EmailResponseState:
        """Step 3: Generate response email using LLM"""
        logger.info("Step 3: Generating email response")
        
        try:
            # Prepare context from search results
            context = ""
            if state['search_results']:
                context = "\n\nRelevant information from knowledge base:\n"
                for i, result in enumerate(state['search_results'], 1):
                    context += f"{i}. {result['data']}\n"
            
            user = state['current_user']
            
            prompt = f"""
            You are {user.name} responding to an email inquiry.

            Original email:
            {state['user_email']}

            Email intent summary:
            {state['email_summary']}
            {context}

            Please generate a professional, helpful email response in JSON format that:
            1. Addresses the sender's specific question or request
            2. Uses the relevant information from the knowledge base if available
            3. Maintains a professional and friendly tone
            4. Includes appropriate email formatting (greeting, body, closing)
            5. Signs off with {user.name}

            Return your response in this exact JSON format:
            {{
            "response_email_subject": "Subject line here...",
            "response_email_body": "Full email body here, including greeting, content, and sign-off with {user.name}."
            }}

            Only return valid JSON. Do not include any explanations or text outside the JSON.
            """
            
            response = self.llm.invoke([HumanMessage(content=prompt)])
            response_json = json.loads(response.content)

            logger.info("Structured email response generated successfully")

            return {
                **state,
                "response_email": response_json
            }
            
        except Exception as e:
            logger.error(f"Error in generate_email_response: {e}")
            return {
                **state,
                "error": f"Failed to generate response: {str(e)}"
            }
    
    def validate_email_response(self, state: EmailResponseState) -> EmailResponseState:
        """Step 4: Validate how well the response addresses the original email"""
        logger.info("Step 4: Validating email response")
        
        try:
            prompt = f"""
            Evaluate how well the following email response addresses the original inquiry.
            
            Original Email:
            {state['user_email']}
            
            Generated Response:
            {state['response_email']}
            
            Please rate the response on a scale of 1-10 based on:
            - How well it addresses the original question/request
            - Relevance and accuracy of information provided
            - Professional tone and clarity
            - Completeness of the response
            
            Provide only a single number between 1 and 10 as your response.
            """
            
            response = self.llm.invoke([HumanMessage(content=prompt)])
            
            # Extract numeric score
            try:
                validation_score = int(response.content.strip())
                if validation_score < 1 or validation_score > 10:
                    validation_score = 5  # Default to middle score if invalid
            except ValueError:
                validation_score = 5  # Default score if parsing fails
            
            logger.info(f"Validation score: {validation_score}/10")
            
            return {
                **state,
                "validation_score": validation_score
            }
            
        except Exception as e:
            logger.error(f"Error in validate_email_response: {e}")
            return {
                **state,
                "validation_score": 0,
                "error": f"Failed to validate response: {str(e)}"
            }
    
    def process_email(self, user_email: str, current_user: CurrentUser) -> EmailResponseState:
        """Main method to process an email through the entire flow"""
        logger.info("Starting email response generation flow")
        
        initial_state = EmailResponseState(
            user_email=user_email,
            current_user=current_user,
            email_summary="",
            search_results=[],
            response_email={
                "response_email_subject": "",
                "response_email_body": ""
            },
            validation_score=0,
            error=""
        )
        
        try:
            final_state = self.graph.invoke(initial_state)
            logger.info("Email response flow completed successfully")
            return final_state
        except Exception as e:
            logger.error(f"Error in email processing flow: {e}")
            return {
                **initial_state,
                "error": f"Flow execution failed: {str(e)}"
            }


In [7]:
def main():
    """Example usage of the email response flow"""
    
    # Initialize the flow
    email_flow = EmailResponseFlow()
    
    # Example user
    current_user = CurrentUser(
        user_id=123,
        name="John Smith",
        email="john.smith@company.com",
    )
    
    # Example incoming email
    user_email = """
    Subject: Consultation Fees and Timing Inquiry

    Hi,

    Could you please confirm Dr. Kiran's consultation fee and available timings on Friday?

    Thank you,  
    Ramesh Patel
    """
    
    # Process the email
    result = email_flow.process_email(user_email, current_user)
    
    # Display results
    print("=" * 60)
    print("EMAIL RESPONSE GENERATION RESULTS")
    print("=" * 60)
    
    if result.get("error"):
        print(f"❌ Error: {result['error']}")
        return
    
    print(f"📧 Original Email Summary:")
    print(f"   {result['email_summary']}")
    print()
    
    print(f"🔍 Search Results Found: {len(result['search_results'])}")
    for i, result_item in enumerate(result['search_results'], 1):
        print(f"   {i}. Score: {result_item['score']:.3f}")
        print(f"      data: {result_item['data'][:100]}...")
    print()
    
    print(f"✉️ Generated Response:")
    print(f"   Subject: {result['response_email']['response_email_subject']}")
    print(f"   Body:")
    print(result['response_email']["response_email_body"])
    print()
    
    print(f"⭐ Validation Score: {result['validation_score']}/10")
    print("=" * 60)

In [8]:
main()

e:\Crail 2025\.venv\Lib\site-packages\langchain_groq\chat_models.py:370: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  warnings.warn(
e:\Crail 2025\.venv\Lib\site-packages\langchain_groq\chat_models.py:370: UserWarning: WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
  warnings.warn(
e:\Crail 2025\.venv\Lib\site-packages\langchain_groq\chat_models.py:370: UserWarning: WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.
  warnings.warn(
INFO:httpx:HTTP Request: GET http://localhost:6334 "HTTP/1.1 200 OK"
INFO:__main__:Connected to Qdrant at localhost:63

EMAIL RESPONSE GENERATION RESULTS
📧 Original Email Summary:
   The sender is inquiring about Dr. Kiran's consultation fee and available timings on Fridays.

🔍 Search Results Found: 5
   1. Score: 0.425
      data: Id: D012 | Name: Dr. Aditya Menon | Designation: Pulmonologist | Specialization: Pulmonology, Sleep ...
   2. Score: 0.413
      data: Id: D006 | Name: Dr. Rohan Patil | Designation: ENT Specialist | Specialization: ENT, Allergy Treatm...
   3. Score: 0.406
      data: Id: D020 | Name: Dr. Varsha Pillai | Designation: Radiologist | Specialization: Radiology, Ultrasoun...
   4. Score: 0.392
      data: Id: D010 | Name: Dr. Vikram Iyer | Designation: Urologist | Specialization: Urology, Kidney Stones |...
   5. Score: 0.386
      data: Id: D004 | Name: Dr. Nikhil Sharma | Designation: Neurologist | Specialization: Neurology, Stroke Ma...

✉️ Generated Response:
   Subject: Re: Consultation Fees and Timing Inquiry
   Body:
Dear Ramesh Patel,

Thank you for your inquiry about Dr.